# EEAI-Chile — Pilot Study: APA7 Descriptive Tables

This notebook builds, in **APA 7th-edition** format, the descriptive tables for the content-validity / pilot study and exports them to a single Word document:

- **Table 1** — Sociodemographic and professional characteristics of participants.
- **Table 2** — Participants' evaluation of the instrument.

Run the cells in order in Google Colab. The Word file is saved to your Drive folder and downloaded to your computer.

In [1]:
# ==========================================================================
# Install dependencies (safe to re-run)
# ==========================================================================
!pip install -q python-docx openpyxl

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 1. Load data

In [2]:
# ==========================================================================
# Configuration and data loading
# ==========================================================================
import re
import numpy as np
import pandas as pd
from IPython.display import display

# --- Paths (edit if needed) ----------------------------------------------
DATA_DIR    = "../data/pilot"
XLSX_PATH   = f"{DATA_DIR}/EEAI-Chile_Pilot_Data_2026_anonymized.xlsx"
OUTPUT_DOCX = "../outputs/tables/EEAI_Demographics_Tables_APA7.docx"

# Read the (first) sheet of the Google Forms export
df = pd.read_excel(XLSX_PATH, sheet_name=0)
C = df.columns
N = len(df)
print(f"Loaded {N} responses, {len(C)} columns.")

# Columns are selected by position (stable for this Forms export).
# Quick check so you can confirm the mapping:
_check = {3:"Contract", 4:"Teaching hours", 5:"Working regime", 6:"OECD area",
          7:"Years exp.", 8:"Degree", 9:"Gender", 10:"Age", 11:"Research",
          12:"Mgmt", 13:"Marital", 14:"Children", 15:"MacArthur",
          50:"Tenure", 51:"Faculty dev.", 52:"Job security",
          46:"Eval: instructions", 47:"Eval: items", 48:"Eval: satisfaction",
          49:"Eval: relevance"}
for i, name in _check.items():
    print(f"  [{i}] {name:24s} <- {str(C[i])[:60]}")

Loaded 104 responses, 57 columns.
  [3] Contract                 <- Tipo de contrato con su institución *
  [4] Teaching hours           <- Número de horas de docencia directa a la semana (en aula o e
  [5] Working regime           <- Régimen de jornada laboral *
  [6] OECD area                <- Área del conocimiento en la que imparte docencia — Clasifica
  [7] Years exp.               <- Número de años de experiencia como docente universitario/a *
  [8] Degree                   <- Grado académico máximo obtenido *
  [9] Gender                   <- Género *
  [10] Age                      <- Edad
  [11] Research                 <- ¿Realiza actividades de investigación actualmente? *
  [12] Mgmt                     <- ¿Ocupa actualmente algún cargo de gestión académica? *
  [13] Marital                  <- Estado civil *
  [14] Children                 <- Número de hijos/as a su cargo *
  [15] MacArthur                <- Estatus social subjetivo — Escala MacArthur *
  [50] Tenure      

## 2. Data preparation

Short variable handles, the derived **Discipline** column (from the OECD/Frascati area code), an optional cleaning step for an implausible tenure value, and English labels for all categories.

In [3]:
# ==========================================================================
# Build a tidy working frame, the Discipline variable, and English labels
# ==========================================================================
d = pd.DataFrame({
    "contract":   df[C[3]],
    "hours":      pd.to_numeric(df[C[4]], errors="coerce"),
    "regime":     df[C[5]],
    "ocde_area":  df[C[6]],
    "years_exp":  pd.to_numeric(df[C[7]], errors="coerce"),
    "degree":     df[C[8]],
    "gender":     df[C[9]],
    "age":        pd.to_numeric(df[C[10]], errors="coerce"),
    "research":   df[C[11]],
    "mgmt":       df[C[12]],
    "marital":    df[C[13]],
    "children":   df[C[14]],
    "macarthur":  pd.to_numeric(df[C[15]], errors="coerce"),
    "tenure":     pd.to_numeric(df[C[50]], errors="coerce"),
    "facdev":     df[C[51]],
    "jobsec":     pd.to_numeric(df[C[52]], errors="coerce"),
    "eval_instr": df[C[46]],
    "eval_items": df[C[47]],
    "eval_satis": df[C[48]],
    "eval_relev": df[C[49]],
})

# --- (4) Discipline: derived from the OECD field of science (Frascati) -----
# The area arrives as a full label, e.g. "5.2 Economía y Negocios"; we read the
# leading major field number (1-6) and map it to the Frascati Manual (2015) field.
OCDE_DISCIPLINE = {
    1: "Natural sciences",
    2: "Engineering and technology",
    3: "Medical and health sciences",
    4: "Agricultural and veterinary sciences",
    5: "Social sciences",
    6: "Humanities and the arts",
}
def to_discipline(label):
    m = re.match(r"\s*(\d+)\.", str(label))
    return OCDE_DISCIPLINE.get(int(m.group(1)), "Other/Unclassified") if m else "Other/Unclassified"
d["discipline"] = d["ocde_area"].apply(to_discipline)

# --- English translations of category labels -------------------------------
TR = {
 "contract": {
   "Adjunto / Por horas de clase": "Adjunct / hourly-paid",
   "Planta (jornada completa o parcial indefinida)": "Tenured/permanent (open-ended)",
   "Contrata (plazo fijo renovable)": "Fixed-term (renewable)",
   "Otro": "Other"},
 "regime": {
   "Solo por horas de clase (sin jornada fija)": "Hourly teaching only (no fixed schedule)",
   "Jornada completa (44 horas semanales)": "Full-time (44 h/week)",
   "Media jornada (22 horas semanales)": "Half-time (22 h/week)",
   "Jornada parcial (menos de 22 horas)": "Part-time (<22 h/week)"},
 "degree": {
   "Licenciado": "Bachelor's",
   "Magíster / Master": "Master's",
   "Doctor/a (Ph.D. o equivalente)": "Doctorate (PhD or equivalent)",
   "Postdoctorado": "Postdoctoral"},
 "gender": {
   "Hombre": "Man", "Mujer": "Woman", "Prefiero no indicar": "Prefer not to say"},
 "research": {
   "No realizo investigación actualmente": "Not currently conducting research",
   "Sí, con recursos propios": "Yes, self-funded",
   "Sí, con financiamiento institucional": "Yes, institutionally funded",
   "Sí, con financiamiento externo (ANID, Fondecyt, etc.)": "Yes, externally funded (ANID, Fondecyt, etc.)"},
 "mgmt": {
   "No": "No",
   "Sí (Director/a de carrera, Decano/a, Jefe/a de área u otro)": "Yes (program director, dean, head of area, or other)"},
 "marital": {
   "Casado/a o en convivencia civil": "Married or in civil partnership",
   "Soltero/a": "Single",
   "Separado/a o divorciado/a": "Separated or divorced",
   "Conviviente (sin acuerdo legal)": "Cohabiting (no legal agreement)",
   "Viudo/a": "Widowed",
   "Prefiero no indicar": "Prefer not to say"},
 "children": {
   "Ninguno": "None", "1 hijo/a": "1", "2 hijos/as": "2", "3 o más hijos/as": "3 or more"},
 "facdev": {
   "Sí, de forma voluntaria": "Yes, voluntarily",
   "Sí, de forma obligatoria": "Yes, as a requirement",
   "No, pero están disponibles en mi institución": "No, but available at my institution",
   "No, no existen ese tipo de programas en mi institución": "No, no such programs exist"},
}
JOBSEC = {1: "Very insecure", 2: "Insecure", 3: "Neither insecure nor secure",
          4: "Secure", 5: "Very secure"}
EVAL   = {"Bajo grado": "Low", "Aceptable grado": "Fair",
          "Buen grado": "Good", "Excelente grado": "Excellent"}

for col, m in TR.items():
    d[col] = d[col].map(lambda x: m.get(x, x))
d["jobsec_lab"] = d["jobsec"].map(JOBSEC)
for c in ["eval_instr", "eval_items", "eval_satis", "eval_relev"]:
    d[c] = d[c].map(lambda x: EVAL.get(x, x))

# Display order for ordinal variables (nominal ones are sorted by frequency)
ORDERS = {
 "degree":     ["Bachelor's", "Master's", "Doctorate (PhD or equivalent)", "Postdoctoral"],
 "children":   ["None", "1", "2", "3 or more"],
 "discipline": list(OCDE_DISCIPLINE.values()),
 "jobsec_lab": [JOBSEC[i] for i in range(1, 6)],
}
print("Discipline counts:\n", d["discipline"].value_counts().to_string())

Discipline counts:
 discipline
Natural sciences                        35
Social sciences                         35
Medical and health sciences             13
Agricultural and veterinary sciences    13
Humanities and the arts                  6
Engineering and technology               2


## 3. APA7 formatting helpers

In [4]:
# ==========================================================================
# APA7 Word helpers: tables (bold "Table N", italic title, horizontal rules
# only, italic "Note.") and a generic table builder.
# ==========================================================================
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

APA_FONT = "Times New Roman"; APA_BODY_SIZE = 11; APA_TABLE_SIZE = 10

def _set_cell_border(cell, **kw):
    tcPr = cell._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = OxmlElement("w:tcBorders"); tcPr.append(b)
    for edge in ("top", "bottom", "left", "right"):
        if edge in kw:
            el = b.find(qn(f"w:{edge}"))
            if el is None:
                el = OxmlElement(f"w:{edge}"); b.append(el)
            for k, v in kw[edge].items():
                el.set(qn(f"w:{k}"), str(v))

def _no_borders(cell):
    _set_cell_border(cell, top={"val": "nil"}, bottom={"val": "nil"},
                     left={"val": "nil"}, right={"val": "nil"})

def _style_run(run, bold=False, italic=False, size=APA_BODY_SIZE, font=APA_FONT):
    run.bold = bold; run.italic = italic
    run.font.size = Pt(size); run.font.name = font
    rPr = run._element.get_or_add_rPr()
    rF = rPr.find(qn("w:rFonts"))
    if rF is None:
        rF = OxmlElement("w:rFonts"); rPr.append(rF)
    for a in ("w:ascii", "w:hAnsi", "w:cs"):
        rF.set(qn(a), font)

def _add_paragraph(doc, segs, align=WD_ALIGN_PARAGRAPH.LEFT,
                   space_before=0, space_after=0, line_spacing=1.0):
    p = doc.add_paragraph(); p.alignment = align
    pf = p.paragraph_format
    pf.space_before = Pt(space_before); pf.space_after = Pt(space_after)
    pf.line_spacing = line_spacing
    for t, s in segs:
        _style_run(p.add_run(t), **s)
    return p

def _set_cell_text(cell, text, bold=False, italic=False,
                   align=WD_ALIGN_PARAGRAPH.LEFT, size=APA_TABLE_SIZE):
    cell.text = ""
    p = cell.paragraphs[0]; p.alignment = align
    p.paragraph_format.space_before = Pt(1)
    p.paragraph_format.space_after = Pt(1)
    p.paragraph_format.line_spacing = 1.0
    _style_run(p.add_run("" if text is None else str(text)),
               bold=bold, italic=italic, size=size)

def _set_col_widths(table, widths):
    table.autofit = False; table.allow_autofit = False
    tbl = table._tbl; tblPr = tbl.tblPr
    lay = tblPr.find(qn("w:tblLayout"))
    if lay is None:
        lay = OxmlElement("w:tblLayout"); tblPr.append(lay)
    lay.set(qn("w:type"), "fixed")
    grid = tbl.find(qn("w:tblGrid"))
    if grid is not None:
        for gc in list(grid.findall(qn("w:gridCol"))):
            grid.remove(gc)
        for w in widths:
            gc = OxmlElement("w:gridCol")
            gc.set(qn("w:w"), str(int(w.inches * 1440))); grid.append(gc)
    for row in table.rows:
        for j, w in enumerate(widths):
            row.cells[j].width = w

def add_apa_table(doc, df_in, table_number, title, note=None, bold_section_col=None,
                  col_aligns=None, table_font_size=APA_TABLE_SIZE,
                  italic_headers=None, col_widths=None):
    """Insert an APA7 table from a DataFrame (columns become the header row)."""
    _add_paragraph(doc, [(f"Table {table_number}", dict(bold=True))],
                   space_before=12, space_after=0)
    _add_paragraph(doc, [(title, dict(italic=True))], space_after=4)
    cols = list(df_in.columns); n_rows = len(df_in) + 1; n_cols = len(cols)
    italic_headers = set(italic_headers or []); col_aligns = col_aligns or {}
    table = doc.add_table(rows=n_rows, cols=n_cols)
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(cols):
        c = table.cell(0, j); _no_borders(c)
        _set_cell_text(c, col, bold=True, italic=(col in italic_headers),
                       align=(WD_ALIGN_PARAGRAPH.CENTER if j > 0 else WD_ALIGN_PARAGRAPH.LEFT),
                       size=table_font_size)
    for i in range(len(df_in)):
        row = df_in.iloc[i]; is_sec = False
        if bold_section_col is not None:
            v = row[bold_section_col]
            is_sec = (isinstance(v, str) and v.strip() != "" and
                      all(str(row[c]).strip() == "" for c in cols if c != bold_section_col))
        for j, col in enumerate(cols):
            c = table.cell(i + 1, j); _no_borders(c)
            al = col_aligns.get(col, WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER)
            _set_cell_text(c, row[col], bold=is_sec, align=al, size=table_font_size)
    line = {"val": "single", "sz": 6, "color": "000000", "space": 0}
    for j in range(n_cols):
        _set_cell_border(table.cell(0, j), top=line, bottom=line)
        _set_cell_border(table.cell(n_rows - 1, j), bottom=line)
    if col_widths is not None:
        _set_col_widths(table, col_widths)
    if note:
        _add_paragraph(doc, [("Note. ", dict(italic=True)), (note, dict())],
                       space_before=4, space_after=6)
    else:
        doc.add_paragraph()

## 4. Table 1 — Sociodemographic and professional characteristics

In [5]:
# ==========================================================================
# Build Table 1
# ==========================================================================
def fmt_pct(freq, base):
    return f"{int(freq)} ({100*freq/base:.1f}%)"

def cat_block(rows, label, series, order=None):
    """Categorical variable -> a bold section row + one n (%) row per category."""
    rows.append([label, "", ""])
    s = series.dropna(); base = len(s); vc = s.value_counts()
    cats = order if order else list(vc.sort_values(ascending=False).index)
    for cat in cats:
        if cat in vc.index:
            rows.append(["", str(cat), fmt_pct(vc[cat], base)])

def num_block(rows, label, series, decimals=1):
    """Continuous variable -> M (SD), Median (Q1, Q3), Range."""
    rows.append([label, "", ""])
    s = series.dropna()
    rows.append(["", "M (SD)", f"{s.mean():.{decimals}f} ({s.std():.{decimals}f})"])
    rows.append(["", "Median (Q1, Q3)",
                 f"{s.median():.{decimals}f} ({s.quantile(.25):.{decimals}f}, {s.quantile(.75):.{decimals}f})"])
    rows.append(["", "Range (min–max)", f"{s.min():g}–{s.max():g}"])

rows = []
cat_block(rows, "Contract type with the institution", d["contract"])
num_block(rows, "Direct teaching hours per week", d["hours"])
cat_block(rows, "Working-time arrangement", d["regime"])
cat_block(rows, "Discipline (OECD field, Frascati Manual)", d["discipline"], ORDERS["discipline"])
num_block(rows, "Years of experience as a university teacher", d["years_exp"])
cat_block(rows, "Highest academic degree", d["degree"], ORDERS["degree"])
cat_block(rows, "Gender", d["gender"])
num_block(rows, "Age (years)", d["age"])
cat_block(rows, "Currently conducting research", d["research"])
cat_block(rows, "Academic management position", d["mgmt"])
cat_block(rows, "Marital status", d["marital"])
cat_block(rows, "Dependent children", d["children"], ORDERS["children"])
num_block(rows, "Subjective social status (MacArthur Scale, 1–10)", d["macarthur"])
num_block(rows, "Years at current institution", d["tenure"])
cat_block(rows, "Participation in faculty development programs (past 2 years)", d["facdev"])
cat_block(rows, "Perceived job continuity next year (1–5)", d["jobsec_lab"], ORDERS["jobsec_lab"])

table1 = pd.DataFrame(rows, columns=["Variable", "Category", f"n = {N}"])
display(table1)

,Variable,Category,n = 104
0,Contract type with the institution,,
1,,Adjunct / hourly-paid,50 (48.1%)
2,,Tenured/permanent (open-ended),46 (44.2%)
3,,Fixed-term (renewable),7 (6.7%)
4,,Other,1 (1.0%)
...,...,...,...
62,,Very insecure,7 (6.7%)
63,,Insecure,8 (7.7%)
64,,Neither insecure nor secure,23 (22.1%)
65,,Secure,36 (34.6%)


## 5. Table 2 — Participants' evaluation of the instrument

In [6]:
# ==========================================================================
# Build Table 2
# ==========================================================================
aspects = [("Comprehension of the instructions", "eval_instr"),
           ("Comprehension of the items",        "eval_items"),
           ("Satisfaction with the instrument",  "eval_satis"),
           ("Relevance of the questions",        "eval_relev")]
levels = ["Low", "Fair", "Good", "Excellent"]

rows2 = []
for label, col in aspects:
    s = d[col].dropna(); base = len(s); vc = s.value_counts()
    rows2.append([label] + [fmt_pct(vc[l], base) if l in vc.index else "0 (0.0%)" for l in levels])

table2 = pd.DataFrame(rows2, columns=["Instrument aspect"] + levels)
display(table2)

,Instrument aspect,Low,Fair,Good,Excellent
0,Comprehension of the instructions,0 (0.0%),7 (6.7%),33 (31.7%),64 (61.5%)
1,Comprehension of the items,2 (1.9%),10 (9.6%),30 (28.8%),62 (59.6%)
2,Satisfaction with the instrument,2 (1.9%),15 (14.4%),40 (38.5%),47 (45.2%)
3,Relevance of the questions,3 (2.9%),9 (8.7%),41 (39.4%),51 (49.0%)


## 6. Export both tables to Word (APA7) and download

In [7]:
# ==========================================================================
# Assemble the Word document and download it
# ==========================================================================
import os

# Edit table numbers/titles to match your manuscript if needed
TABLE1_NUMBER = 4
TABLE2_NUMBER = 7

doc = Document()
nm = doc.styles["Normal"]; nm.font.name = APA_FONT; nm.font.size = Pt(APA_BODY_SIZE)

add_apa_table(
    doc, table1, TABLE1_NUMBER,
    "Sociodemographic and Professional Characteristics of Participants",
    note=(f"N = {N}. Values are n (%) for categorical variables and M (SD), median (Q1, Q3), "
          "and range for continuous variables. Discipline was derived from the OECD field of "
          "science and technology (Frascati Manual, 2015) based on the area reported by each "
          "participant. Percentages are computed on valid responses for each variable."),
    bold_section_col="Variable",
    col_widths=[Inches(3.0), Inches(2.3), Inches(1.2)],
)

add_apa_table(
    doc, table2, TABLE2_NUMBER,
    "Participants’ Evaluation of the Instrument",
    note=(f"N = {N}. Values are n (%). Response options were ordered from Low to Excellent. "
          "Percentages are computed on valid responses for each aspect."),
    col_widths=[Inches(2.6), Inches(0.95), Inches(0.95), Inches(0.95), Inches(0.95)],
)

# Save to outputs directory
doc.save(OUTPUT_DOCX)
print(f"✅ Saved: {OUTPUT_DOCX}")


✅ Saved: ../outputs/tables/EEAI_Demographics_Tables_APA7.docx
